# WSI classification with bounded attention MIL

Use the ten local **TCGA-LUAD / TCGA-LUSC** slides to assemble one small bag per WSI and train an attention-based multiple instance learning (MIL) classifier. A frozen ImageNet DINO encoder embeds each selected patch; the attention pool and classification head learn from **slide-level labels**, not invented patch labels.

**Workflow:** verified WSI + tissue coordinates → bounded in-memory patch bag → frozen encoder → attention pooling → one class probability vector per WSI.

This is a pipeline demonstration with **10 patients, 8 patches per slide and one training epoch**. Its two-slide held-out result is not a benchmark, clinical validation, full-tissue inference, or evidence of generalization. The train/validation/test split is a reproducible teaching split, not an official TCGA evaluation protocol.


## 1. Setup and explicit resource bounds

Restart the kernel before running all cells. Use the project's main `.venv` kernel after `uv sync --locked --extra train` (include `--extra pretrained` too if using the segmentation notebook). The TCGA slides, a complete calibrated preprocessing run, and DINO's previously downloaded checkpoint must already be local. This notebook performs no downloads, copies no WSIs and writes no individual patch images.

Paths are discovered from the checkout or `notebooks/` directory; set `PROJECT` explicitly when launched elsewhere. Review the cohort paths and settings before running. All artifacts go into a fresh ignored output directory. The training run has a five-minute wall-clock bound; automated notebook checks additionally bound each code cell.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import sys
import time
import uuid

for variable in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[variable] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

PROJECT = None  # Optional: Path(...) to this checkout.
if PROJECT is None:
    PROJECT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / 'pyproject.toml').is_file() and (p / 'src/train.py').is_file()), None)
    if PROJECT is None:
        raise FileNotFoundError('Open from this project or set PROJECT explicitly.')
PROJECT = Path(PROJECT).expanduser().resolve(strict=True)
sys.path.insert(0, str(PROJECT))
os.environ['HF_HOME'] = str(PROJECT / 'data/models/huggingface')
os.environ['HF_HUB_CACHE'] = str(PROJECT / 'data/models/huggingface/hub')
from huggingface_hub import constants as hf_constants
assert hf_constants.HF_HUB_OFFLINE is True, 'Restart the kernel and run all cells to enable offline model loading.'
assert Path(hf_constants.HF_HUB_CACHE).resolve() == (PROJECT / 'data/models/huggingface/hub').resolve(), 'Restart the kernel and run all cells to select the project model cache.'

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

CLASS_NAMES = ['LUAD', 'LUSC']
SEED = 42
PATCHES_PER_SLIDE = 8
TARGET_MPP = 0.5
ROSTER = PROJECT / 'manifests/tcga_demo.csv'
PREPROCESSING_RUN = PROJECT / 'outputs/tcga-mpp05'
assert 1 <= PATCHES_PER_SLIDE <= 8
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUT = PROJECT / 'outputs/wsi_classification_demo' / run_id
OUT.mkdir(parents=True, exist_ok=False)
started = time.monotonic()
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

def sha256(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

print('Kernel:', sys.executable)
print('Outputs:', OUT)


## 2. Assign patients to splits before reading patches

Each local slide has a declared participant and LUAD/LUSC project label. This demo requires five distinct participants per class and one slide per participant. Within each class, a seeded permutation assigns three participants to training, one to validation and one to test: **6 / 2 / 2 patients**, with both labels represented in each split.

The source roster records provider checksums. This bounded notebook checks file size and metadata/coordinate identity; it does not rehash every gigabyte of WSI data. Declared provider checksums remain distinct from hashes actually computed in this run.


In [ ]:
with ROSTER.open(newline='') as stream:
    roster = list(csv.DictReader(stream))
preprocessing_manifest_path = PREPROCESSING_RUN / 'manifest.json'
preprocessing = json.loads(preprocessing_manifest_path.read_text())
assert preprocessing['status'] == 'complete'
assert len(roster) == len(preprocessing['slides']) == 10
slides = {Path(slide['input']).stem: slide for slide in preprocessing['slides']}
assert len(slides) == 10 and len({row['case_submitter_id'] for row in roster}) == 10
assert len({row['case_id'] for row in roster}) == 10
assert all(row['case_submitter_id'].strip() and row['case_id'].strip() for row in roster)
rng = np.random.default_rng(SEED)
assignments = {}
for label in CLASS_NAMES:
    cohort = sorted((row for row in roster if row['label'] == label), key=lambda row: row['case_submitter_id'])
    assert len(cohort) == 5
    ordered = [cohort[i] for i in rng.permutation(len(cohort))]
    for row, split in zip(ordered, ['train', 'train', 'train', 'val', 'test'], strict=True):
        assignments[row['case_submitter_id']] = split
rows, source_state = [], {}
for row in sorted(roster, key=lambda row: (row['label'], row['case_submitter_id'])):
    slide = slides[row['file_id']]
    wsi = (PROJECT / row['path']).resolve(strict=True)
    assert wsi == (PROJECT / slide['input']).resolve(strict=True)
    assert slide['name'] == row['case_submitter_id'] and slide['case_id'] == row['case_id']
    assert slide['label'] == row['label']
    assert wsi.stat().st_size == int(row['file_size'])
    layouts = [item for item in slide['exports']['layouts']
               if item['patch_size'] == 256 and item.get('target_mpp') == TARGET_MPP]
    assert len(layouts) == 1
    layout = layouts[0]
    assert layout['exported_count'] == layout['eligible_count'] > 0
    coords_path = (PREPROCESSING_RUN / layout['coords']).resolve(strict=True)
    rows.append({'wsi_path': str(wsi), 'coords_path': str(coords_path),
        'split': assignments[row['case_submitter_id']], 'group_id': row['case_submitter_id'],
        'label': CLASS_NAMES.index(row['label']), 'bag_id': row['file_id'],
        'case_id': row['case_id'], 'source_md5': row['md5sum'], 'source_size_bytes': row['file_size']})
    source_state[row['file_id']] = {'path': str(wsi), 'size_bytes': wsi.stat().st_size,
        'mtime_ns': wsi.stat().st_mtime_ns, 'provider_md5_declared': row['md5sum'],
        'full_wsi_checksum_verified_this_run': False, 'coords_path': str(coords_path),
        'coords_sha256': sha256(coords_path)}
manifest = OUT / 'wsi_splits.csv'
with manifest.open('x', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
cohort_table = pd.DataFrame(rows)
assert cohort_table.groupby('group_id')['split'].nunique().max() == 1
assert cohort_table.groupby('bag_id')['split'].nunique().max() == 1
assert cohort_table['split'].value_counts().to_dict() == {'train': 6, 'val': 2, 'test': 2}
cohort_table['diagnosis'] = cohort_table['label'].map(dict(enumerate(CLASS_NAMES)))
display(cohort_table[['group_id', 'bag_id', 'diagnosis', 'split', 'source_size_bytes']])
display(pd.crosstab(cohort_table['split'], cohort_table['diagnosis']).reindex(['train', 'val', 'test']))
print('Preprocessing method:', preprocessing['mask_method'])
print('Each slide uses its full eligible 256-tile coordinate file; only 8 positions will be sampled.')


## 3. Compose Hydra settings and inspect the selected tissue positions

The notebook calls the same `src.train.run` function as the Hydra CLI. Its WSI data adapter reads **eight deterministic, evenly spaced indices from each exported coordinate list**, then reads those regions directly into memory. This is spatial sampling of an existing list, not exhaustive tissue coverage or a tumour-selection rule.

The selected local export is the recorded **`clam_saturation` / schema-5 CLAM-style** run. It is not silently reinterpreted as the newer CODAIT → TIA → CLAM pipeline. Coordinates retain their source calibration: output tiles are 256 pixels at target 0.5 µm/pixel, with actual rounded effective MPP retained. DINO subsequently resizes them to 224 pixels; its input sampling is therefore approximately 0.571 µm/pixel, not 0.5.

Gold rectangles below mark the sampled positions on existing small slide thumbnails. No new patch PNGs or WSI copies are written.


In [ ]:
from hydra import compose, initialize_config_dir
from hydra.utils import instantiate
from omegaconf import OmegaConf
import torch
import lightning as L
from src.train import run as run_training

TRAINING = OUT / 'training'
with initialize_config_dir(config_dir=str(PROJECT / 'configs'), version_base='1.3'):
    cfg = compose(config_name='train', overrides=[
        'experiment=mil_dino_attention', 'data=wsi',
        'class_names=[LUAD,LUSC]', f'seed={SEED}', 'num_threads=1',
        f'data.manifest={json.dumps(str(manifest))}', f'data.data_root={json.dumps(str(PROJECT))}',
        'data.batch_size=1', 'data.num_workers=0', 'data.augment=false',
        f'data.max_patches_per_bag={PATCHES_PER_SLIDE}', 'data.image_size=224',
        'data.max_pixels=1000000', f'data.target_mpp={TARGET_MPP}',
        'trainer.accelerator=cpu', 'trainer.devices=1', 'trainer.max_epochs=1',
        'trainer.max_time=00:00:05:00', 'trainer.num_sanity_val_steps=0',
        'trainer.limit_train_batches=1.0', 'trainer.limit_val_batches=1.0', 'trainer.limit_test_batches=1.0',
        'trainer.enable_progress_bar=false', 'trainer.enable_model_summary=false',
        f'trainer.default_root_dir={json.dumps(str(TRAINING))}',
        f'callbacks.checkpoint.dirpath={json.dumps(str(TRAINING / "checkpoints"))}',
        'callbacks.checkpoint.save_last=false', f'logger.save_dir={json.dumps(str(TRAINING))}',
        'test=true',
    ])
assert cfg.model.encoder.name == 'vit_small_patch16_224.dino'
assert cfg.model.encoder.pretrained and cfg.model.encoder.freeze
assert cfg.model.pooling == 'attention' and cfg.data.task == 'mil'
torch.set_num_threads(1)
if torch.get_num_interop_threads() != 1:
    torch.set_num_interop_threads(1)  # Restart the kernel before changing thread policy.
data = instantiate(cfg.data, _convert_='all')
data.setup('fit')
data.setup('test')
assert {split: len(dataset) for split, dataset in data.datasets.items()} == {'train': 6, 'val': 2, 'test': 2}
selections = data.selections
assert len(selections) == 10
(OUT / 'sampled_coordinates.json').write_text(json.dumps(selections, indent=2, allow_nan=False))
OmegaConf.save(cfg, OUT / 'notebook_config.yaml', resolve=True)
print(OmegaConf.to_yaml(cfg, resolve=True))


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7), layout='constrained')
for ax, row in zip(axes.flat, rows, strict=True):
    slide = slides[row['bag_id']]
    selection = selections[row['bag_id']]
    thumbnail = PREPROCESSING_RUN / slide['folder'] / 'thumbnail.png'
    assert sha256(thumbnail) == slide['thumbnail_sha256']
    with Image.open(thumbnail) as image:
        assert image.width * image.height <= 4_000_000
        preview = np.asarray(image.convert('RGB'))
    width, height = selection['dimensions']
    sw, sh = selection['source_size']
    ax.imshow(preview, extent=(0, width, height, 0))
    for x, y in selection['coords']:
        ax.add_patch(Rectangle((x, y), sw, sh, fill=False, edgecolor='gold', linewidth=1.2))
    ax.set_title(f"{row['group_id']}\n{CLASS_NAMES[row['label']]} · {row['split']} · {len(selection['coords'])} sampled")
    ax.axis('off')
fig.suptitle('Selected tissue positions for bounded WSI bags — not full-slide inference')
fig.savefig(OUT / 'sampled_tissue_positions.png')
plt.show()


## 4. Verify the local pretrained encoder, then train the MIL head

DINO here is **ImageNet-pretrained ViT-S/16**, not a pathology-specific foundation model and not UNI. Its existing local safetensors checkpoint is identified by revision, size and SHA-256 before use. Offline Hugging Face settings prevent an implicit download.

The encoder remains frozen and in evaluation mode. Only the attention network and slide classifier learn. Training performs six optimizer steps (one pass over six bags), selects its checkpoint using two validation slides, and evaluates the held-out pair once through the training entrypoint. Do not tune settings using this pair; a different research study needs a larger locked split and independent evaluation.


In [ ]:
DINO_REVISION = '10e440b8a34dfd657a90f2fcaa988c6b6a2f0da4'
DINO_SHA256 = '8d4795b8e4e327c6d841e15a2e3e6d88c45346256a43f8cdae8c26d92c806f7a'
DINO_CACHE = PROJECT / 'data/models/huggingface/hub/models--timm--vit_small_patch16_224.dino'
assert (DINO_CACHE / 'refs/main').read_text().strip() == DINO_REVISION
encoder_weights = (DINO_CACHE / 'snapshots' / DINO_REVISION / 'model.safetensors').resolve(strict=True)
assert encoder_weights.is_relative_to(PROJECT / 'data/models')
assert encoder_weights.stat().st_size == 86_676_340
assert sha256(encoder_weights) == DINO_SHA256
encoder_provenance = {'model': cfg.model.encoder.name, 'training_domain': 'ImageNet-1k',
    'revision': DINO_REVISION, 'weights_path': str(encoder_weights), 'sha256': DINO_SHA256,
    'bytes': encoder_weights.stat().st_size, 'frozen': True, 'offline': True}
(OUT / 'encoder_provenance.json').write_text(json.dumps(encoder_provenance, indent=2))
fit_started = time.monotonic()
training_result = run_training(cfg, TRAINING)
assert training_result['status'] == 'complete' and training_result['global_step'] == 6
assert training_result['samples_by_split'] == {'train': 6, 'val': 2, 'test': 2}
assert all(np.isfinite(value) for value in training_result['metrics'].values())
print(f"Completed six optimizer steps in {time.monotonic() - fit_started:.1f} seconds.")
display(pd.DataFrame([training_result['metrics']]))


## 5. Reload the selected checkpoint and inspect one prediction per held-out WSI

Reloading includes the trained attention/head and the frozen encoder weights. It does not request the pretrained encoder again. Model and data contracts must match the recorded checkpoint. The following prediction pass uses the **same two test bags and coordinates**, not additional test examples.

Softmax values below are uncalibrated model scores after a tiny training run. Their sum is one by construction; that does not establish reliability. The confusion table illustrates slide-level bookkeeping and is too small for a performance claim.


In [ ]:
checkpoint = Path(training_result['checkpoint']).resolve(strict=True)
assert checkpoint.is_relative_to(TRAINING.resolve())
model_cfg = OmegaConf.to_container(cfg.model, resolve=True)
model_cfg['encoder']['pretrained'] = False  # All weights come from this run's checkpoint.
model = instantiate(model_cfg, _convert_='all')
state = torch.load(checkpoint, map_location='cpu', weights_only=False)  # Trusted checkpoint created above.
assert state['model_contract'] == model.contract
assert state['data_contract'] == data.contract
model.load_state_dict(state['state_dict'], strict=True)
model.eval()
assert all(not parameter.requires_grad for parameter in model.encoder.parameters())
assert not model.encoder.training
predictor = L.Trainer(accelerator='cpu', devices=1, logger=False,
    enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
    limit_predict_batches=2)
batches = predictor.predict(model, datamodule=data)
held_out = {row['bag_id']: row for row in rows if row['split'] == 'test'}
prediction_rows = []
for batch in batches:
    probabilities = batch['probability'].detach().cpu().numpy()
    assert probabilities.shape[1] == len(CLASS_NAMES) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0)
    for bag_id, scores in zip(batch['id'], probabilities, strict=True):
        row = held_out[bag_id]
        prediction_rows.append({'slide_id': bag_id, 'patient_id': row['group_id'],
            'reference_label': CLASS_NAMES[row['label']], 'predicted_label': CLASS_NAMES[int(scores.argmax())],
            'score_LUAD': float(scores[0]), 'score_LUSC': float(scores[1]),
            'sampled_patches': len(selections[bag_id]['coords'])})
assert len(prediction_rows) == len(held_out) == 2
assert {row['slide_id'] for row in prediction_rows} == set(held_out)
predictions = pd.DataFrame(prediction_rows)
predictions.to_csv(OUT / 'held_out_slide_predictions.csv', index=False)
display(predictions)
confusion = pd.crosstab(predictions['reference_label'], predictions['predicted_label']).reindex(
    index=CLASS_NAMES, columns=CLASS_NAMES, fill_value=0)
confusion.to_csv(OUT / 'held_out_confusion_counts.csv')
display(confusion)
fig, ax = plt.subplots(figsize=(7, 3.5), layout='constrained')
positions = np.arange(len(predictions))
ax.barh(positions, predictions['score_LUAD'], label='LUAD score')
ax.barh(positions, predictions['score_LUSC'], left=predictions['score_LUAD'], label='LUSC score')
ax.set_yticks(positions, [f"{row.patient_id} (reference {row.reference_label})" for row in predictions.itertuples()])
ax.set_xlim(0, 1)
ax.set_xlabel('Uncalibrated model score')
ax.set_title('Two held-out WSI bags · mechanics demo only')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), ncol=2)
fig.savefig(OUT / 'held_out_slide_scores.png')
plt.show()


## 6. Save the evidence and keep the interpretation bounded

The receipt binds the source roster, preprocessing manifest, sampled coordinates, encoder/checkpoint hashes, composed settings and outputs. WSI size/mtime checks are repeated; they are not full-file checksum verification. Source images and coordinate files are read-only, and no individual patches were exported.

For a research experiment, replace this teaching split with prespecified patient-disjoint cohorts, justify slide and patch sampling, measure uncertainty at the intended prediction unit, and audit sites/scanners and pretrained-data overlap. Attention weights are aggregation weights, not validated tumour annotations. See [training](../docs/training.md) and [evaluation considerations](../docs/computational-pathology-considerations.md).


In [ ]:
for bag_id, source in source_state.items():
    current = Path(source['path']).stat()
    assert current.st_size == source['size_bytes'] and current.st_mtime_ns == source['mtime_ns']
    assert sha256(source['coords_path']) == source['coords_sha256']
assert sha256(encoder_weights) == DINO_SHA256
assert len(list((TRAINING / 'checkpoints').glob('*.ckpt'))) == 1
receipt = {'status': 'complete', 'scope': 'bounded real-WSI classification mechanics',
    'class_names': CLASS_NAMES, 'seed': SEED, 'patients': 10, 'slides': 10,
    'patients_by_split': {'train': 6, 'val': 2, 'test': 2},
    'max_patches_per_slide': PATCHES_PER_SLIDE, 'target_read_mpp': TARGET_MPP,
    'encoder_input_pixels': 224, 'encoder_input_sampling_note': 'Effective tile MPP multiplied by 256/224 after resizing',
    'source_roster': str(ROSTER), 'source_roster_sha256': sha256(ROSTER),
    'preprocessing_manifest': str(preprocessing_manifest_path),
    'preprocessing_manifest_sha256': sha256(preprocessing_manifest_path),
    'preprocessing_method': preprocessing['mask_method'], 'source_files': source_state,
    'source_sizes_mtimes_unchanged': True, 'coordinate_hashes_unchanged': True,
    'split_manifest_sha256': sha256(manifest), 'sampled_coordinates': selections,
    'encoder': encoder_provenance, 'checkpoint': str(checkpoint), 'checkpoint_sha256': sha256(checkpoint),
    'training': training_result, 'predictions': prediction_rows,
    'device': 'cpu', 'torch_threads': torch.get_num_threads(),
    'no_individual_patch_images_written': True,
    'source_code_sha256': {name: sha256(PROJECT / name) for name in (
        'src/data/wsi.py', 'src/tiling.py', 'src/train.py', 'src/models/vision.py', 'src/models/encoders.py')},
    'elapsed_seconds': time.monotonic() - started,
    'interpretation': '10-patient, one-epoch, sampled-patch demo; not full-slide inference or a generalization/clinical benchmark'}
(OUT / 'run_receipt.json').write_text(json.dumps(receipt, indent=2, allow_nan=False))
print('Complete: 10 real WSIs, 6 training steps, 2 validation and 2 test bags.')
print('Artifacts:', OUT)
